In [ ]:
%load_ext autoreload
%autoreload 2


from kontext.visualization import space_plot,space_plot_mask,get_figs,get_proportions,plot_region_distribution,heatmap_proportions
from utils.load_data import load_and_preprocess_adata
from kontext.kontext_class import Kontext
from kontext.utils import lr_progression_df,get_LR_pairs

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
plt.rcParams['figure.facecolor'] = '#fcf8f500'
plt.rcParams['axes.facecolor'] = '#fcf8f500'

from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

## Supplementary table 4

In [ ]:
ct3D = {
    'CM':'Cardiomyocyte',
    'FB':'Fibroblast',
    'FBmyo':'Myofibroblast',
    'Macro':'Macrophage',
    'mo':'Myeloid_other',
    'DC':'Myeloid_other',
    'Mast':'Myeloid_other',
    'Neutrophil':'Myeloid_other',
    'B':'Lymphoid',
    'ILC2':'Lymphoid',
    'MAIT':'Lymphoid',
    'NK':'Lymphoid',
    'T':'Lymphoid',
    'Treg':'Lymphoid',
    'Fibrocytes':'Fibrocyte',
    'Erythrocyte':'Other',
    'NM':'Other',
    'Schwann':'Other',
    'vEC':'vEC',
    'vEpicardial':'vEpicardial',
    'vPericyte':'vPericyte',
    'vSMC':'vSMC',
    'vSMC1':'vSMC',
    'vSMC2':'vSMC'}

pd.DataFrame(ct3D.items(),columns=['original','refined']).to_csv('./kontext_reproducibility/figures/lymph/table_cell_types.csv',index=True)

## Load dataset and compute spatial domains

In [ ]:

data = {'dataset':'LYMPH','sample':'3d'}  # Info to load the dataset 
preprocessing = {'preprocessing':'sparse_hvg_LR'} # Preprocessing params 
embedding = {"sigma_e":1, "snn":20, 'alpha':0.5} # Embedding params 
spatial_domains = {'clustering':'gmm','seed':0,'n_domains':2} # Spatial domains params

adata = load_and_preprocess_adata(data,preprocessing_params=preprocessing)
ko = Kontext(**embedding,verbose=True) # Init Kontext object 
ko.spatial_domains(adata,**spatial_domains) # Compute the spatial domains 


## Description of the dataset

In [ ]:
print(f"Cell types in presence :{adata.obs['annotation'].unique().tolist()}")
print(f"Shape of the dataset : {adata.shape}")
print(f"Number of sections : {adata.obs['n_section'].nunique()}")
print(f"Sections : {list(adata.obs['n_section'].unique())}")
print(f"Spatial coordinates : \
\n x in [{adata.obsm['spatial'][:,0].min():.2f},{adata.obsm['spatial'][:,0].max():.2f}]\
\n y in [{adata.obsm['spatial'][:,1].min():.2f},{adata.obsm['spatial'][:,1].max():.2f}]\
\n z in [{adata.obsm['spatial'][:,2].min():.2f},{adata.obsm['spatial'][:,2].max():.2f}]")

## Cell types and domain colors 

In [ ]:
s2 = 'spatial_domains'
adata.uns['annotation_colors'] = [
    '#3B75AF',
    '#EF8636',
    '#C53A32',
    '#9E47F3',
    '#84584E',
    '#B7BD6D',
    '#58BBCC',
    '#B3C6E5',
    '#F5BE82',
    '#A8DD93',
    '#F19D99',
]
adata.uns[f'{s2}_colors'] = [ "#FFAD41","#B34A44"]

# Reorient spatial coordinate to align them with the 3D plots
adata.obsm['spatial'] = adata.obsm['spatial']*(-1,-1,1)



## Boundary definition

In [ ]:

ko.niche(adata, cell_type_key=s2)
conds = [(adata.obs[s2]==0)&(adata.obs[f'niche_{s2}_1']==0), 
        (adata.obs[s2]==0)&(adata.obs[f'niche_{s2}_1']>0), 
        (adata.obs[s2]==1)&(adata.obs[f'niche_{s2}_0']==0), 
        (adata.obs[s2]==1)&(adata.obs[f'niche_{s2}_0']>0)
        ]
adata.obs[f'region_{s2}'] = np.select(conds, ['TME', 'boundary', 'tumor', 'boundary'])
adata.uns[f'region_{s2}_colors'] = [ "#FFAD41","#1BA99B","#B34A44"]


## Figure 4B  - Space plot of the cell types

In [ ]:
sections = adata.obs['n_section'].unique()
prev_ = {sections[i]: sections[i-1] if i>0 else sections[i] for i in range(len(sections))}
section = 34

fig,ax  = get_figs(1,5,5)
adata_ = adata[adata.obs['n_section'].isin([section])].copy() 
mask = space_plot_mask(adata_, 'n_section', section,color='annotation', ax=ax, snm=1, sm=30)
ax.set_title(f"Cell types",fontsize=15)
ax.set_xlabel("")
ax.set_ylabel(f"Section {section}",fontsize=15)
ax.legend(fontsize=15,frameon=False,bbox_to_anchor=[1,1])

## Space plot of the spatial domains

In [ ]:
section = 34
adata_ = adata[adata.obs['n_section'].isin([section])].copy() 
fig,ax  = get_figs(1,5,5)
mask = space_plot_mask(adata_, 'n_section', section,color=f'{s2}', ax=ax, snm=1, sm=30)
ax.set_title(f"Kontext Regions",fontsize=15)
ax.set_xlabel("")
ax.set_ylabel("")
ax.get_legend().remove()

## Figure 4D - Space plot of the regions 

In [ ]:
section = 34
adata_ = adata[adata.obs['n_section'].isin([section])].copy() 
fig,ax  = get_figs(1,5,5)
mask = space_plot_mask(adata_, 'n_section', section,color=f'region_{s2}', ax=ax, snm=1, sm=30)
ax.set_title(f"Kontext Regions",fontsize=15)
ax.set_xlabel("")
ax.set_ylabel("")
ax.get_legend().remove()

## Supplementary figure S11

In [ ]:
fig,ax  = get_figs(1,10,5)
proportions = get_proportions(adata,'annotation',f'{s2}',normalize=False)
plot_region_distribution(proportions,colors = adata.uns[f'{s2}_colors'] ,ax=ax,fontsize=12,threshold=.1)
print(proportions)
ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=15) 
ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 
ax.set_title('Region proportions',fontsize=15)
ax.set_xlabel('Normalized proportions',fontsize=15)
ax.legend(fontsize=15,bbox_to_anchor=(1.2,1))
fig.tight_layout()
plt.show()

## Figure 4E

In [ ]:
fig,ax  = get_figs(1,10,5)
proportions = get_proportions(adata,'annotation',f'region_{s2}',normalize=False)
plot_region_distribution(proportions,colors = adata.uns[f'region_{s2}_colors'] ,ax=ax,fontsize=12,threshold=.1)
print(proportions)
ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=15) 
ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 
ax.set_title('Region proportions',fontsize=15)
ax.set_xlabel('Normalized proportions',fontsize=15)
fig.tight_layout()
ax.legend(fontsize=15,bbox_to_anchor=(1.5,1))
plt.show()

## Figure 4C

### Generate the 3D plots 

In [ ]:
import pyvista as pv
from scipy.spatial import cKDTree

coords = adata.obsm['spatial']
region_codes = adata.obs[f'region_{s2}'].astype('category').cat.codes.values
tree = cKDTree(coords)
_, nn_idx = tree.query(coords, k=2) 
nn_idx = nn_idx[:, 1]

region_3Dplot = {'boundary':{'color':"#1BA99B",'opacity': 1,'point_size':1},
                #  'TME':{'color':"#FFAD41",'opacity':0.02,'point_size':6},
                 'tumor':{'color':"#B34A44",'opacity':1,'point_size':6},}

pl = pv.Plotter()
for region,v in region_3Dplot.items():

    boundary_mask = adata.obs[f'region_{s2}']==region
    boundary_coords = coords[boundary_mask]

    cloud_boundary = pv.PolyData(boundary_coords)
    pl.add_mesh(
        cloud_boundary,
        color=v['color'],
        render_points_as_spheres=True,
        point_size=6,
        opacity=v['opacity'],
    )

pl.add_axes()
pl.show()



### Show viewpoints of interest.

In [ ]:
print(pl.camera_position)

pl.camera_position = [(-23314, -22001,10469),(-8396, -4083, 550),(0.0, 0.0, 1.0)]
pl.show()
pl.camera_position = [(-23314, -22001,10469),(-8396, -4083, 550),(0.0, 0.0, 1.0)]
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot1.tif',transparent_background=True)


print(pl.camera_position)
pl.camera_position = [(-12053, -20052, 5000), (-10015, -4015, -1000),(0.0, 0.0, 1.0)] 
pl.show()
pl.camera_position = [(-12053, -20052, 5000), (-10015, -4015, -1000),(0.0, 0.0, 1.0)] 
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot2.tif',transparent_background=True)

## Figure 4F - Sender-receiver communication scores 

### Initiate LR pairs 

In [ ]:
pairs = get_LR_pairs(adata, organism='human', verbose=False)
genes = pairs.index.tolist()
mode = 'mean'
# lr_normalize(adata, pairs)

### Init communication scores 

In [ ]:
from kontext.utils import lr_normalize,lr_moments,lr_availability,lr_moments,lr_score,lr_aggregate,lr_cell_score,lr_log
import seaborn as sns
groups, ts, tr = 'annotation', 'CAF', 'Tumor cell'
mode, samples_ = 'mean', ['TME', 'boundary', 'tumor']
# mode, times_ = 'mean', ['sham', 'cryo7D', 'cryo28D']  

genes = pairs.index.tolist()
mode = 'mean'

pairs = get_LR_pairs(adata, organism='human', verbose=False)
ko.cell_cell_communication(adata,pairs,mode='mean')



groups = 'annotation'
t1, t2 = 'CAF', 'Tumor cell'
for receiver,sender in (t1, t2),(t2,t1),(t1, t1), (t2, t2): 
    ko.init_sender_receiver_communication(adata,pairs,sender,receiver,groups)


### Plot LR scores heatmaps 

In [ ]:

send_recep = [(t1, t2),(t2,t1),(t1, t1), (t2, t2) ]
top = 30

genes_of_interest = [
'ADM_ACKR3', 
'CXCL8_SDC1', 
'ANGPTL4_SDC4', 
'ANGPTL4_SDC1', 
'VEGFA_EGFR', 
'VEGFA_CD44', 
'VEGFA_ITGB1', 
'VEGFC_ITGB1',

'ICAM4_ITGB1',
'F13A1_ITGB1',
# 'PRG4_CD44',
'CCL5_SDC1',
'HGF_SDC1',
'PTN_SDC2',
'HGF_SDC2',

'C3_CD46',

'JAG1_NOTCH3', 
'DLL1_NOTCH3',

'LGALS9_MET',
'ADAM10_MET',
'ADAM17_MET',
'SEMA5A_MET',
'SEMA4D_MET',
'DCN_MET',
'HGF_MET',
'LRIG1_MET',
'TIMP3_MET',]

genes = pairs.index.tolist()
fig, axes = plt.subplots(ncols=4,nrows=1,figsize=(24,5))
Ss = {}
i = -1
for ax, (s_, r_) in enumerate(send_recep ):
    i+=1
    print(i//2,i%2)
    ax = axes[i]
    r= {'col':groups,'value':r_}

    if f'{s_}{r_}' in Ss:
        S = Ss[f'{s_}{r_}']
    else:
        S = lr_score(adata, s_, r_, groups, mode,r=r,src='P')
        Ss[f'{s_}{r_}'] = S
    print(s_,r_,S.shape)

    gi = pd.Index(adata.uns['lr']['genes']) # genes 
    pos = adata.obs_names.get_indexer(S.index) # positions of the obs of interest 
    det = pd.Series(np.asarray((adata.obsm['X_lr'][pos][:, gi.get_indexer(pairs.receptor)] > 0).mean(0)).ravel(),index=pairs.index) # percentage of cells expressing the receptors
    print('det',det[det.index.isin(genes_of_interest)])
    keep = det[det > .15].index#.intersection(prev.columns[(prev > .05).any()])   # receptor + ligand support
    T = lr_aggregate(adata, S, s_, by=f'region_{s2}')['all'].reindex(samples_)[[g for g in genes if (g in keep and g in S.columns)]]# or 'MET' in g]]

    print(s_,r_)
    exp = (lr_log(T).max())
    T = T[exp[exp>0.1].index]
    Rt1 = (lr_log(T).loc['tumor'] - lr_log(T).loc['boundary'])
    Rt2 = (lr_log(T).loc['boundary']-lr_log(T).loc['tumor'])
    final_genes = Rt1.nlargest(top).index.union(Rt2.nlargest(top).index) 

    print(len(final_genes))
    T = T[[g for g in genes_of_interest if g in T.columns]]
    Rt1 = (lr_log(T).loc['tumor'] - lr_log(T).loc['boundary'])
    T = T[Rt1.nlargest(len(T.T)).index]
    T = T.T[['boundary','tumor']].T
    print(lr_log(T).T)
    cbar = i==3
    hm = sns.heatmap((lr_log(T) if mode == 'mean' else T).T, ax=ax, cmap='RdBu_r', center=0, vmin=-2,vmax=2, 
                annot=True, fmt='.2f', annot_kws={'fontsize':15},
                linewidths=.5,cbar=cbar, cbar_kws={'label': 'Sender-receiver score' if mode == 'mean' else 'score'}) 
    if cbar:
        cbar_ax = hm.collections[0].colorbar
        cbar_ax.set_label(cbar_ax.ax.get_ylabel(), fontsize=20)


    ax.set_title(f'{s_} → {r_}', fontsize=20) 
    ax.set_xlabel('') 
    ylabel = 'LR pairs' if i==0 else ""
    ax.set_ylabel(ylabel,fontsize=15) 
    ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=20) 
    ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 

fig.tight_layout() 
plt.show() 

## Supplementary figure S10 and S11

In [ ]:
sections_ = sections
nrows = 4
ncols= 5
figsize = ncols*4 ,nrows * 5
for color in ['annotation','spatial_domains']:
    fig,axes = plt.subplots(ncols=ncols,nrows =nrows,figsize = figsize)
    for i,section in enumerate(np.sort(sections_)[::-1]): 
        adata_ = adata[adata.obs[f'n_section']==section].copy()
        adata_.obs['spatial_domains'] = adata_.obs['spatial_domains'].astype('category')
        ax = axes[i//5,i%5]
        space_plot(adata_,color,ax=ax)
        ax.set_title(f'section {section}',fontsize=15)
        ax.set_ylabel('',fontsize=15)
        ax.set_xlabel('',fontsize=15)
        ax.get_legend().remove()
    fig.tight_layout()
    axes[3,-1].set_axis_off()
    plt.show()

## Supplementary figure S13

### Preparation of the groups to plot and their colors

In [ ]:

cell_types = adata.obs['annotation'].sort_values().unique().tolist()
colors = adata.uns['annotation_colors']
regions = 'boundary','tumor','TME'
domains = 0,1


regions_3Dplot_ct = {}
for r in regions :
    for t,c in zip(cell_types,colors):
        regions_3Dplot_ct[f'{t}_{r}'] = {'color':c,'opacity':1,'point_size':1,'t':t,'r':r}

domains_3Dplot_ct = {}

for r in domains:
    for t,c in zip(cell_types,colors):
        domains_3Dplot_ct[f'{t}_{r}'] = {'color':c,'opacity':1,'point_size':1,'t':t,'r':r}
display(domains_3Dplot_ct)

### 3D plot generation and point of view

In [ ]:
import pyvista as pv
from scipy.spatial import cKDTree



coords = adata.obsm['spatial']
region_codes = adata.obs[f'region_{s2}'].astype('category').cat.codes.values
tree = cKDTree(coords)
_, nn_idx = tree.query(coords, k=2) 
nn_idx = nn_idx[:, 1]


adata.obs[f'annotation_region'] = pd.Categorical(adata.obs['annotation'].astype(str) + '_' + adata.obs[f'region_{s2}'].astype(str))
adata.obs[f'annotation_domain'] = pd.Categorical(adata.obs['annotation'].astype(str) + '_' + adata.obs[f'{s2}'].astype(str))
# print(adata.obs[f'annotation_domain'])



pl = pv.Plotter()
for region,v in regions_3Dplot_ct.items():
    print(region,v)

    if v['t'] in ['CAF','CAM','Keratin Pearl',"Tumor cell","Plasma IgG"]:
        if v['r'] == 'tumor':
            boundary_mask = adata.obs[f'annotation_region']==region
            print(region,boundary_mask.sum())
            boundary_coords = coords[boundary_mask]

            cloud_boundary = pv.PolyData(boundary_coords)
            pl.add_mesh(
                cloud_boundary,
                color=v['color'],
                render_points_as_spheres=True,
                point_size=6,
                opacity=v['opacity'],
            )

pl.add_axes()
pl.show()

pl.camera_position = [(-23314, -22001,-10469),(-8396, -4083, 550),(0.0, 0.0, -1.0)]
pl.show()
pl.camera_position = [(-23314, -22001,-10469),(-8396, -4083, 550),(0.0, 0.0, -1.0)]
pl.show()


print(pl.camera_position)
pl.camera_position = [(-4053, -20052, -5000), (-10015, -4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
pl.camera_position = [(-4053, -20052, -5000), (-10015, -4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()





## Supplementary figure S14

### Enumeration of the domain of assignation of the cell type

In [ ]:
import pandas as pd
import seaborn as sns

cell_type_col = 'annotation'   # <-- replace with your actual cell-type column name

records = []

for i, section in enumerate(np.sort(sections)):
    adata_ = adata[adata.obs[f'n_section'] == section].copy()
    adata_.obs['spatial_domains_3D'] = adata_.obs['spatial_domains'].astype('category')
    adata_.uns['spatial_domains_3D_colors'] = adata_.uns['spatial_domains_colors']
    ko_section = Kontext(**embedding, verbose=True)
    ko_section.spatial_domains(adata_, **spatial_domains)
    ko_section.weights(adata_)

    align_binary_columns(adata_, 'spatial_domains_3D', 'spatial_domains')
    spatial_domains_2D += [adata_.obs['spatial_domains']]

    for ct in adata_.obs[cell_type_col].unique():
        mask = adata_.obs[cell_type_col] == ct
        records.append({
            'cell_type': ct,
            'section': section,
            '3D': adata_.obs.loc[mask, 'spatial_domains_3D'].astype(int).mean(),
            '2D': adata_.obs.loc[mask, 'spatial_domains'].astype(int).mean(),
        })



### 

### Plot of the assignation discrepancy between 2D and 3D 

In [ ]:
df = pd.DataFrame(records)
df_long = df.melt(
    id_vars=['cell_type', 'section'],
    value_vars=['3D', '2D'],
    var_name='spatial_domains',
    value_name='proportion_domain_1'
)
print(df_long)
df_long = df_long.loc[df_long['cell_type'].isin(['Plasma IgG','Fibroblast','CAF','Macrophage'])]
fig, ax = plt.subplots(figsize=(5, 5))
sns.boxplot(data=df_long, x='cell_type', y='proportion_domain_1', hue='spatial_domains', ax=ax,palette={'3D': "#e75fa3", '2D': "#201588"},)
ax.set_ylabel('Proportion in domain 1')
ax.set_xlabel('Cell type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Supplementary figure S15

In [ ]:
def align_binary_columns(adata, col1, col2, diff_col='diff'):
    c1 = adata.obs[col1].astype(int)
    c2 = adata.obs[col2].astype(int)

    # Flip col2 only if it currently disagrees with col1 more than it agrees
    flip = (c1 == c2).mean() < 0.5
    c2_aligned = 1 - c2 if flip else c2

    adata.obs[col2] = pd.Categorical(c2_aligned)

    adata.obs[diff_col] = (c1 != c2_aligned).astype(int)


sections_diff = [2,4,18,19,23,33,]



ncols = 3
nrows=len(sections_diff)
# nrows=len(sections)
figsize = ncols*7 ,nrows * 4

fig,axes = plt.subplots(ncols=ncols,nrows =nrows,figsize = figsize)

spatial_domains_2D = []
for i,section in enumerate(np.sort(sections_diff)): 
# for i,section in enumerate(np.sort(sections)): 

    adata_ = adata[adata.obs[f'n_section']==section]
    adata_.obs['spatial_domains_3D'] = adata_.obs['spatial_domains'].astype('category')
    adata_.uns['spatial_domains_3D_colors'] = adata_.uns['spatial_domains_colors']
    ko_section = Kontext(**embedding,verbose=True)
    ko_section.spatial_domains(adata_,**spatial_domains)
    ko_section.weights(adata_)

    align_binary_columns(adata_,'spatial_domains_3D','spatial_domains')
    spatial_domains_2D += [adata_.obs['spatial_domains']]

    ax = axes[i,0]
    space_plot(adata_,'spatial_domains_3D',ax=ax)
    title = f'3D spatial_domains' if i == 0 else ''
    ax.set_title(title,fontsize=25)
    ax.set_xlabel('',fontsize=15)
    ax.set_ylabel(f'section {section}',fontsize=25)
    # ax.get_legend().remove()


    ax = axes[i,1]
    space_plot(adata_,'spatial_domains',ax=ax)
    title = f'2D spatial_domains' if i == 0 else ''
    ax.set_title(title,fontsize=25)
    ax.set_xlabel('',fontsize=25)
    ax.set_ylabel('',fontsize=25)
    # ax.set_ylabel(f'section {section}',fontsize=25)

    ax = axes[i,2]
    space_plot(adata_,'diff',ax=ax)
    title = f'Difference \n {int(adata_.obs['diff'].sum())/len(adata_):.2f}% different' if i == 0 else f'{int(adata_.obs['diff'].sum())/len(adata_):.2f}% different'
    ax.set_title(title,fontsize=25)
    ax.set_xlabel('',fontsize=15)
    ax.set_ylabel('',fontsize=15)

fig.tight_layout()
plt.show()


## Supplementary Figure S16

In [ ]:
def pie_region(adata, value, region_col, col='annotation', ax=None, colors=None):
    p = adata.obs.loc[adata.obs[col] == value, region_col].value_counts()#.reindex(['TME','boundary','tumor'])
    print(value,p)
    ax = ax or get_figs(1, 5, 5)[1]
    ax.pie(p.values, labels=[f'{k}\n{v:,}' for k, v in p.items()], autopct='%1.0f%%',
           colors=colors or adata.uns.get(f'{region_col}_colors'), startangle=90)
    ax.set_title(f'{value}  (n={p.sum():,})', fontsize=13)

fig, axes = get_figs(2, 5, 5)
for ax, t in zip(axes, [ts, tr]): pie_region(adata, t, f'{s2}', groups, ax=ax)
fig, axes = get_figs(2, 5, 5)
for ax, t,colors in zip(axes, [ts, tr],[[ '#1BA99B', '#B34A44','#FFAD41',],[  '#B34A44','#1BA99B','#FFAD41',]]): pie_region(adata, t, f'region_{s2}', groups, ax=ax,colors=colors)